In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="04-youtube-video-search",
    notebook_name="02_multimodal_embeddings.ipynb"
)

# Multimodal Embeddings for Video Search

## The Big Idea (For a 12-Year-Old)

Imagine you have a video of a dog barking at a mailman. That single video has **three types of information**:
- **What you SEE**: A dog, a mailman, a front yard (visual)
- **What you HEAR**: Barking sounds, maybe the mailman saying "easy boy" (audio)
- **What is WRITTEN**: The title says "My Dog vs The Mailman LOL" (text)

Each of these is a different **modality** -- a different way of understanding the video. "Multi-modal" means using ALL of these together, not just one.

Think of it like describing a movie to a friend. You could describe what you saw, what the music sounded like, and what the characters said. Each description gives a different piece of the puzzle. Using all three together gives the best picture.

In this notebook, we will build each piece and then learn how to **fuse** them into a single embedding that captures everything about the video.

---

## Staff-Level Technical Summary

We implement:
- **Text embeddings**: From BoW/TF-IDF baselines to BERT-style contextual encoders
- **Visual features**: Frame sampling, CNN/ViT feature extraction, temporal pooling
- **Audio features**: Mel spectrograms, audio classification embeddings
- **Fusion strategies**: Early fusion, late fusion, cross-attention
- **Shared embedding space**: Projecting all modalities into a single vector space for cross-modal retrieval

## 1. Text Embeddings: From Simple to Powerful

### The Evolution

Text embeddings have evolved over time, like going from a bicycle to a car to a rocket ship:

1. **Bag of Words (BoW)**: Count how many times each word appears. Simple, but ignores word order ("dog bites man" = "man bites dog").
2. **TF-IDF**: Like BoW, but rare words get more weight. The word "the" appears everywhere (low weight), but "tensorflow" is rare (high weight).
3. **Word2Vec**: Learns that "king" and "queen" are related. Each word becomes a dense vector.
4. **BERT/Transformers**: The same word gets DIFFERENT embeddings based on context. "bank" in "river bank" vs "bank account" = different vectors. This is the state of the art.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# === Method 1: Bag of Words ===

def bag_of_words(documents):
    """
    Convert documents to Bag-of-Words vectors.
    
    12-year-old version:
    For each document, count how many times each word appears.
    'I like cats and cats like me' -> {'I':1, 'like':2, 'cats':2, 'and':1, 'me':1}
    Then convert to a vector where each position = one word.
    
    Problem: 'Dog bites man' and 'Man bites dog' get the same vector!
    """
    # Build vocabulary
    vocab = set()
    for doc in documents:
        vocab.update(doc.lower().split())
    vocab = sorted(vocab)
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    
    # Build vectors
    vectors = np.zeros((len(documents), len(vocab)))
    for i, doc in enumerate(documents):
        for word in doc.lower().split():
            vectors[i, word_to_idx[word]] += 1
    
    return vectors, vocab


# Demo
documents = [
    "funny cat video compilation",
    "cat playing with laser pointer",
    "dog training tutorial basics",
    "machine learning tutorial python",
    "funny dog fails compilation"
]

bow_vectors, vocab = bag_of_words(documents)

print("=== Bag of Words ===")
print(f"\nVocabulary ({len(vocab)} words): {vocab}")
print(f"\nVector shape: {bow_vectors.shape} (documents x vocab_size)")
print(f"\nSparse? Most entries are zero:")
print(f"  Non-zero entries: {np.count_nonzero(bow_vectors)} / {bow_vectors.size} = {np.count_nonzero(bow_vectors)/bow_vectors.size*100:.1f}%")

# Show similarity
from numpy.linalg import norm
def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

print(f"\nCosine similarities (BoW):")
print(f"  'funny cat video' vs 'cat playing laser':   {cosine_sim(bow_vectors[0], bow_vectors[1]):.3f}")
print(f"  'funny cat video' vs 'dog training':        {cosine_sim(bow_vectors[0], bow_vectors[2]):.3f}")
print(f"  'funny cat video' vs 'funny dog fails':     {cosine_sim(bow_vectors[0], bow_vectors[4]):.3f}")
print(f"  'dog training' vs 'ML tutorial':            {cosine_sim(bow_vectors[2], bow_vectors[3]):.3f}")
print(f"\nProblem: BoW thinks 'funny cat' is more similar to 'funny dog' than to 'cat playing'")
print(f"because they share the word 'funny' and 'compilation'. It misses semantic meaning.")

In [ ]:
# === Method 2: TF-IDF ===

def compute_tf_idf(documents):
    """
    TF-IDF: Term Frequency * Inverse Document Frequency.
    
    12-year-old version:
    Words that appear A LOT in ONE document but rarely in OTHER documents
    are the most important. The word 'the' appears everywhere (low IDF),
    so it gets low weight. 'TensorFlow' appears in just one doc (high IDF),
    so it gets high weight.
    
    TF  = (count of word in doc) / (total words in doc)
    IDF = log(total docs / docs containing the word)
    TF-IDF = TF * IDF
    """
    # Build vocabulary
    vocab = set()
    tokenized = [doc.lower().split() for doc in documents]
    for tokens in tokenized:
        vocab.update(tokens)
    vocab = sorted(vocab)
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    
    n_docs = len(documents)
    
    # Compute IDF for each word
    idf = np.zeros(len(vocab))
    for i, word in enumerate(vocab):
        doc_count = sum(1 for tokens in tokenized if word in tokens)
        idf[i] = np.log(n_docs / (doc_count + 1))  # +1 to avoid division by zero
    
    # Compute TF-IDF
    tfidf_vectors = np.zeros((n_docs, len(vocab)))
    for i, tokens in enumerate(tokenized):
        tf = Counter(tokens)
        total = len(tokens)
        for word, count in tf.items():
            j = word_to_idx[word]
            tfidf_vectors[i, j] = (count / total) * idf[j]
    
    return tfidf_vectors, vocab, idf


tfidf_vectors, vocab_tfidf, idf = compute_tf_idf(documents)

print("=== TF-IDF ===")
print(f"\nIDF values (higher = more rare/important):")
word_idf = [(w, idf[i]) for i, w in enumerate(vocab_tfidf)]
word_idf.sort(key=lambda x: -x[1])
for w, val in word_idf[:8]:
    print(f"  '{w}': {val:.3f}")

print(f"\nShared words (low IDF, appear in many docs):")
for w, val in word_idf[-5:]:
    print(f"  '{w}': {val:.3f}")

print(f"\nCosine similarities (TF-IDF):")
print(f"  'funny cat video' vs 'cat playing laser':   {cosine_sim(tfidf_vectors[0], tfidf_vectors[1]):.3f}")
print(f"  'funny cat video' vs 'funny dog fails':     {cosine_sim(tfidf_vectors[0], tfidf_vectors[4]):.3f}")
print(f"  'dog training' vs 'ML tutorial':            {cosine_sim(tfidf_vectors[2], tfidf_vectors[3]):.3f}")
print(f"\nTF-IDF is better than BoW because it downweights common words,")
print(f"but it STILL does not understand that 'tutorial' and 'lesson' mean the same thing.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# === Method 3: Learned Embeddings (Word2Vec-style) ===

class SimpleWordEmbedding(nn.Module):
    """
    Learn dense word vectors from co-occurrence patterns.
    
    12-year-old version:
    Instead of hand-counting words, we let the computer LEARN
    what each word means by looking at which words appear together.
    
    'Cat' and 'kitten' appear in similar contexts -> similar vectors.
    'Cat' and 'Python' appear in different contexts -> different vectors.
    
    Each word becomes a short (e.g., 64-dim) dense vector instead of
    a long sparse vector.
    """
    def __init__(self, vocab_size, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
    
    def forward(self, token_ids):
        embeds = self.embedding(token_ids)  # (batch, seq_len, embed_dim)
        return embeds.mean(dim=1)           # Average pool -> (batch, embed_dim)


# Demo: Compare sparse vs dense
vocab_size = 50
embed_model = SimpleWordEmbedding(vocab_size, embed_dim=64)

print("=== Sparse vs Dense Embeddings ===")
print(f"\n  BoW/TF-IDF (sparse):")
print(f"    Vector size: {len(vocab_tfidf)} (vocabulary size)")
print(f"    Most values: 0 (sparse)")
print(f"    Example: [0, 0, 0.3, 0, 0, 0, 0.5, 0, 0, 0, ...]")
print(f"\n  Word2Vec / Embeddings (dense):")
print(f"    Vector size: 64 (chosen, usually 64-768)")
print(f"    Every value is non-zero")
print(f"    Example: [0.12, -0.45, 0.78, 0.03, -0.91, ...]")

# Show that similar inputs get similar dense embeddings
torch.manual_seed(42)
# Simulate two similar queries (share tokens)
query_a = torch.tensor([[1, 5, 10, 0, 0]])   # "cat video funny" + padding
query_b = torch.tensor([[1, 5, 12, 0, 0]])    # "cat video hilarious" + padding
query_c = torch.tensor([[20, 30, 40, 0, 0]])   # "quantum physics theory" + padding

emb_a = embed_model(query_a).detach()
emb_b = embed_model(query_b).detach()
emb_c = embed_model(query_c).detach()

sim_ab = F.cosine_similarity(emb_a, emb_b).item()
sim_ac = F.cosine_similarity(emb_a, emb_c).item()

print(f"\n  Cosine similarity (shared context):  {sim_ab:.4f}")
print(f"  Cosine similarity (different topics): {sim_ac:.4f}")
print(f"\n  Even before training, shared tokens create some similarity.")
print(f"  After training on real data, semantically similar words")
print(f"  cluster together automatically.")

In [ ]:
# === Method 4: Transformer-based Encoder (BERT-style) ===

class MiniTransformerEncoder(nn.Module):
    """
    A simplified Transformer encoder for text.
    
    12-year-old version:
    This is the smartest text brain. It reads the WHOLE sentence
    and decides what each word means BASED ON THE OTHER WORDS.
    
    For example:
    - 'I deposited money at the bank' -> 'bank' = financial institution
    - 'I sat by the river bank' -> 'bank' = edge of a river
    Same word, different meaning! BERT figures this out.
    
    Staff-level detail:
    - Self-attention: each token attends to all other tokens
    - Multi-head attention: multiple 'perspectives' on the relationships
    - Layer normalization + feed-forward: post-attention processing
    - [CLS] token pooling: first token's output as sentence embedding
    """
    def __init__(self, vocab_size, embed_dim=128, n_heads=4, n_layers=2, max_seq_len=32):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(max_seq_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_projection = nn.Linear(embed_dim, 256)  # Project to shared embedding space
    
    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        
        # Token + positional embeddings
        positions = torch.arange(seq_len).unsqueeze(0).expand(batch_size, -1)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        
        # Transformer encoding
        x = self.transformer(x)  # (batch, seq_len, embed_dim)
        
        # Pool: use [CLS] token (first position) or mean pooling
        pooled = x.mean(dim=1)  # Mean pooling
        
        # Project to shared space
        output = self.output_projection(pooled)
        output = F.normalize(output, p=2, dim=1)
        
        return output


# Instantiate
text_encoder = MiniTransformerEncoder(vocab_size=5000, embed_dim=128, n_heads=4, n_layers=2)

# Demo forward pass
torch.manual_seed(42)
sample_tokens = torch.randint(1, 5000, (4, 12))  # batch of 4 queries, max 12 tokens each
text_embeddings = text_encoder(sample_tokens)

print("=== Mini Transformer Text Encoder ===")
print(f"\n  Architecture:")
print(f"    Vocab size: 5,000")
print(f"    Embed dim: 128")
print(f"    Attention heads: 4")
print(f"    Transformer layers: 2")
print(f"    Output dim: 256")
print(f"    Total params: {sum(p.numel() for p in text_encoder.parameters()):,}")
print(f"\n  Input:  (batch_size=4, seq_len=12) token IDs")
print(f"  Output: (batch_size=4, 256) L2-normalized embeddings")
print(f"  Shape:  {text_embeddings.shape}")
print(f"  Norm:   {text_embeddings.norm(dim=1).tolist()}")

print(f"\n  In production (BERT-base):")
print(f"    Vocab: 30,522 | Embed: 768 | Heads: 12 | Layers: 12")
print(f"    Params: 110 million")
print(f"    Why so big? BERT was pre-trained on ALL of Wikipedia + BooksCorpus.")

## 2. Visual Features: Understanding Video Content

### The Photo Album Analogy

A 5-minute video at 30 fps has **9,000 frames**. Processing all of them is too slow and wasteful (adjacent frames are nearly identical). Instead, we:

1. **Sample a few frames** (like flipping through a photo album and picking 8 key photos)
2. **Feed each frame through a vision model** (ViT or CNN) to get a "description" (embedding)
3. **Aggregate all frame embeddings** into a single video embedding

There are two main approaches for the vision model:
- **CNN (ResNet, etc.)**: Like a detective with a magnifying glass that scans the image in small patches
- **ViT (Vision Transformer)**: Cuts the image into patches and looks at ALL patches at once (like seeing the whole picture)

In [ ]:
# === Frame Sampling Strategies ===

def uniform_sampling(total_frames, n_samples):
    """Sample frames evenly spaced through the video."""
    indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
    return indices


def random_sampling(total_frames, n_samples):
    """Sample frames randomly."""
    return np.sort(np.random.choice(total_frames, n_samples, replace=False))


def keyframe_sampling(total_frames, n_samples, scene_changes=None):
    """
    Sample at scene change boundaries.
    
    12-year-old version:
    Instead of picking frames at regular intervals, pick frames
    right when the scene CHANGES. That way you capture the most
    diverse content from the video.
    """
    if scene_changes is None:
        # Simulate scene changes at random points
        scene_changes = np.sort(np.random.choice(total_frames, n_samples * 2, replace=False))
    # Pick frames right after scene changes
    return scene_changes[:n_samples]


# Visualize sampling strategies
total_frames = 300  # 10 seconds at 30 fps
n_samples = 8
np.random.seed(42)

strategies = {
    'Uniform Sampling': uniform_sampling(total_frames, n_samples),
    'Random Sampling': random_sampling(total_frames, n_samples),
    'Keyframe Sampling': keyframe_sampling(total_frames, n_samples),
}

fig, axes = plt.subplots(3, 1, figsize=(14, 5), sharex=True)

colors = ['#2196F3', '#4CAF50', '#FF9800']
for ax, (name, frames), color in zip(axes, strategies.items(), colors):
    # Draw timeline
    ax.axhline(y=0, color='gray', linewidth=1)
    ax.scatter(frames, [0] * len(frames), s=200, color=color, zorder=5, edgecolors='black', linewidth=1.5)
    
    for f in frames:
        ax.annotate(f'{f}', (f, 0), textcoords="offset points", xytext=(0, 12),
                   ha='center', fontsize=8)
    
    ax.set_ylabel(name, fontsize=10, fontweight='bold')
    ax.set_ylim(-0.5, 0.5)
    ax.set_yticks([])
    ax.set_xlim(-10, total_frames + 10)

axes[-1].set_xlabel('Frame Index (0 to 300)', fontsize=11)
plt.suptitle('Frame Sampling Strategies for a 10-Second Video (300 frames)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Uniform: Evenly spaced -- simple, reliable, most commonly used.")
print("Random: Can capture unexpected moments but less predictable.")
print("Keyframe: Targets scene changes -- best diversity but requires scene detection.")
print(f"\nOur choice: Uniform sampling -- simple, effective, and deterministic.")

In [ ]:
# === Visual Encoder: CNN vs ViT ===

class SimpleCNNEncoder(nn.Module):
    """
    A simplified CNN for frame encoding (like a mini-ResNet).
    
    12-year-old version:
    This is like a detective with a magnifying glass. It slides
    the glass across the image in small patches, looking for
    edges, then shapes, then objects. Each layer sees bigger patterns.
    
    Layer 1: 'I see edges and corners'
    Layer 2: 'Those edges form a circle -- maybe an eye?'
    Layer 3: 'There is a face with eyes and a nose -- it is a cat!'
    """
    def __init__(self, output_dim=256):
        super().__init__()
        self.features = nn.Sequential(
            # Input: 3 x 64 x 64
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # -> 32 x 32 x 32
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # -> 64 x 16 x 16
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # -> 128 x 8 x 8
            nn.AdaptiveAvgPool2d((1, 1))  # -> 128 x 1 x 1
        )
        self.projection = nn.Linear(128, output_dim)
    
    def forward(self, x):
        # x: (batch, 3, 64, 64)
        features = self.features(x).squeeze(-1).squeeze(-1)  # (batch, 128)
        output = self.projection(features)                    # (batch, output_dim)
        return F.normalize(output, p=2, dim=1)


class SimplePatchEncoder(nn.Module):
    """
    A simplified Vision Transformer (ViT) for frame encoding.
    
    12-year-old version:
    Instead of sliding a magnifying glass, ViT cuts the image into
    a grid of small patches (like a jigsaw puzzle). Then it looks
    at ALL patches at once and figures out how they relate.
    
    It is like looking at all puzzle pieces simultaneously and
    understanding the whole picture, rather than one piece at a time.
    """
    def __init__(self, image_size=64, patch_size=8, embed_dim=128, n_heads=4, output_dim=256):
        super().__init__()
        self.patch_size = patch_size
        n_patches = (image_size // patch_size) ** 2  # 64/8 = 8, 8*8 = 64 patches
        patch_dim = 3 * patch_size * patch_size      # 3 * 8 * 8 = 192
        
        self.patch_embedding = nn.Linear(patch_dim, embed_dim)
        self.position_embedding = nn.Embedding(n_patches, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim * 4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.output_projection = nn.Linear(embed_dim, output_dim)
    
    def forward(self, x):
        # x: (batch, 3, 64, 64)
        batch_size = x.shape[0]
        
        # Extract patches: unfold the image into patch_size x patch_size chunks
        p = self.patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)   # (B, 3, H/p, W/p, p, p)
        patches = patches.contiguous().view(batch_size, -1, 3 * p * p)  # (B, n_patches, patch_dim)
        n_patches = patches.shape[1]
        
        # Embed patches + positions
        positions = torch.arange(n_patches).unsqueeze(0).expand(batch_size, -1)
        x = self.patch_embedding(patches) + self.position_embedding(positions)
        
        # Transformer
        x = self.transformer(x)  # (batch, n_patches, embed_dim)
        
        # Pool and project
        pooled = x.mean(dim=1)  # (batch, embed_dim)
        output = self.output_projection(pooled)
        return F.normalize(output, p=2, dim=1)


# Compare
cnn_encoder = SimpleCNNEncoder(output_dim=256)
vit_encoder = SimplePatchEncoder(output_dim=256)

# Forward pass with random images
torch.manual_seed(42)
dummy_images = torch.randn(4, 3, 64, 64)  # batch of 4 images

cnn_output = cnn_encoder(dummy_images)
vit_output = vit_encoder(dummy_images)

print("=== CNN vs ViT Comparison ===")
print(f"\n  CNN Encoder:")
print(f"    Params: {sum(p.numel() for p in cnn_encoder.parameters()):,}")
print(f"    Output: {cnn_output.shape}")
print(f"    Approach: Slide convolutional filters across the image")
print(f"    Strength: Captures local patterns (edges, textures)")
print(f"\n  ViT Encoder:")
print(f"    Params: {sum(p.numel() for p in vit_encoder.parameters()):,}")
print(f"    Output: {vit_output.shape}")
print(f"    Approach: Split image into 64 patches, self-attention over all")
print(f"    Strength: Captures global relationships between patches")
print(f"\n  Our choice: ViT -- better for search because it captures global context.")
print(f"  A cat in the corner of the image relates to a toy on the other side.")

In [ ]:
# === Complete Video Encoder: Frame Sampling + Encoding + Aggregation ===

class VideoEncoder(nn.Module):
    """
    Full video encoding pipeline:
    1. Take pre-extracted frame features (from ViT)
    2. Apply temporal attention (optional) to weight frames
    3. Aggregate into a single video embedding
    
    Staff-level detail:
    We use a lightweight temporal transformer on top of frame features
    rather than processing raw pixels. This lets us pre-compute frame
    features offline and only run the temporal aggregation during indexing.
    """
    def __init__(self, frame_dim=256, hidden_dim=256, output_dim=256, n_heads=4):
        super().__init__()
        
        # Temporal attention: lets the model weight important frames higher
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=frame_dim, num_heads=n_heads, batch_first=True
        )
        self.layer_norm = nn.LayerNorm(frame_dim)
        self.projection = nn.Sequential(
            nn.Linear(frame_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, frame_features):
        # frame_features: (batch, n_frames, frame_dim)
        
        # Temporal self-attention: each frame attends to all other frames
        attended, attention_weights = self.temporal_attention(
            frame_features, frame_features, frame_features
        )
        attended = self.layer_norm(attended + frame_features)  # Residual connection
        
        # Average pool across frames
        pooled = attended.mean(dim=1)  # (batch, frame_dim)
        
        # Project to output space
        output = self.projection(pooled)
        output = F.normalize(output, p=2, dim=1)
        
        return output, attention_weights


# Demo
video_encoder = VideoEncoder(frame_dim=256, output_dim=256)

# Simulate frame features for a batch of 4 videos, each with 8 frames
torch.manual_seed(42)
frame_features = torch.randn(4, 8, 256)  # 4 videos, 8 frames each, 256-dim features

video_embeddings, attn_weights = video_encoder(frame_features)

print("=== Video Encoder ===")
print(f"\n  Input: {frame_features.shape} (batch=4, frames=8, dim=256)")
print(f"  Output: {video_embeddings.shape} (batch=4, dim=256)")
print(f"  Attention weights: {attn_weights.shape} (batch=4, frames=8, frames=8)")
print(f"  Params: {sum(p.numel() for p in video_encoder.parameters()):,}")

# Visualize attention weights for video 0
fig, ax = plt.subplots(figsize=(8, 6))
attn = attn_weights[0].detach().numpy()
im = ax.imshow(attn, cmap='Blues', aspect='auto')
ax.set_xlabel('Key Frame', fontsize=12)
ax.set_ylabel('Query Frame', fontsize=12)
ax.set_title('Temporal Attention Weights (Video 0)\nWhich frames attend to which?', fontsize=13)
ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_xticklabels([f'F{i}' for i in range(8)])
ax.set_yticklabels([f'F{i}' for i in range(8)])
plt.colorbar(im, label='Attention Weight')
plt.tight_layout()
plt.show()

print("Each row shows how much one frame 'pays attention' to all other frames.")
print("Brighter = more attention. This lets the model focus on important frames.")

## 3. Audio Features: The Third Modality

### Why Audio Matters

Imagine searching for "acoustic guitar cover". The visual frames might just show a person sitting in a room. Without audio, the system cannot tell if they are playing guitar or just talking. Audio adds:

- **Speech content**: What people say in the video (speech-to-text transcription)
- **Music type**: Acoustic guitar vs electric, jazz vs rock
- **Sound effects**: Barking, explosions, laughter -- all add context

The standard approach is to convert audio to a **mel spectrogram** -- a picture of the sound -- and then process it like an image.

In [ ]:
# === Audio Features: Mel Spectrogram ===

def generate_synthetic_audio(duration_sec=3, sample_rate=16000, frequencies=[440, 880]):
    """
    Generate synthetic audio signal.
    
    12-year-old version:
    Sound is a wave. Different sounds have different wave patterns.
    A low note has slow waves (low frequency), a high note has fast waves.
    We combine a few frequencies to simulate a simple sound.
    """
    t = np.linspace(0, duration_sec, int(sample_rate * duration_sec))
    signal = np.zeros_like(t)
    for freq in frequencies:
        signal += np.sin(2 * np.pi * freq * t)
    # Add some noise
    signal += 0.1 * np.random.randn(len(t))
    return signal, sample_rate


def compute_mel_spectrogram(signal, sample_rate, n_mels=64, n_fft=1024, hop_length=512):
    """
    Compute mel spectrogram from audio signal.
    
    12-year-old version:
    A spectrogram is like taking a photo of sound.
    - X-axis = time
    - Y-axis = frequency (pitch)
    - Color = how loud that frequency is
    
    'Mel' scale makes it match how humans hear.
    We hear the difference between 100Hz and 200Hz easily,
    but 10000Hz and 10100Hz sound the same to us.
    Mel scale adjusts for this.
    """
    # Simple spectrogram using STFT (Short-Time Fourier Transform)
    n_frames = (len(signal) - n_fft) // hop_length + 1
    spectrogram = np.zeros((n_fft // 2 + 1, n_frames))
    
    window = np.hanning(n_fft)
    for i in range(n_frames):
        start = i * hop_length
        frame = signal[start:start + n_fft] * window
        spectrum = np.abs(np.fft.rfft(frame))
        spectrogram[:, i] = spectrum
    
    # Simplified mel filter bank
    freq_bins = spectrogram.shape[0]
    mel_filters = np.zeros((n_mels, freq_bins))
    mel_points = np.linspace(0, freq_bins - 1, n_mels + 2, dtype=int)
    for i in range(n_mels):
        start, center, end = mel_points[i], mel_points[i+1], mel_points[i+2]
        mel_filters[i, start:center] = np.linspace(0, 1, center - start)
        mel_filters[i, center:end] = np.linspace(1, 0, end - center)
    
    mel_spec = mel_filters @ spectrogram
    mel_spec_db = 10 * np.log10(mel_spec + 1e-10)  # Convert to dB
    
    return mel_spec_db


# Generate and visualize
np.random.seed(42)
signal_music, sr = generate_synthetic_audio(3, 16000, [440, 880, 1320])  # Musical chord
signal_speech, sr = generate_synthetic_audio(3, 16000, [150, 300, 500])  # Speech-like

mel_music = compute_mel_spectrogram(signal_music, sr)
mel_speech = compute_mel_spectrogram(signal_speech, sr)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im0 = axes[0].imshow(mel_music, aspect='auto', origin='lower', cmap='magma')
axes[0].set_title('Mel Spectrogram: Music (Guitar Chord)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Time Frame')
axes[0].set_ylabel('Mel Frequency Bin')
plt.colorbar(im0, ax=axes[0], label='Power (dB)')

im1 = axes[1].imshow(mel_speech, aspect='auto', origin='lower', cmap='magma')
axes[1].set_title('Mel Spectrogram: Speech-like Signal', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Time Frame')
axes[1].set_ylabel('Mel Frequency Bin')
plt.colorbar(im1, ax=axes[1], label='Power (dB)')

plt.suptitle('Audio -> Mel Spectrogram ("A Photo of Sound")', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Music has energy at higher frequencies (brighter bands at top).")
print("Speech is concentrated at lower frequencies (brighter at bottom).")
print("\nThis 2D image can be fed into a CNN or ViT just like a regular image!")
print("That is the beauty of mel spectrograms -- they turn audio into a vision problem.")

In [ ]:
# === Audio Encoder ===

class AudioEncoder(nn.Module):
    """
    Encodes mel spectrogram into a dense embedding.
    
    12-year-old version:
    Takes the 'photo of sound' (mel spectrogram) and reads it
    like a regular image using a CNN. The output is a short
    summary of what the audio sounds like.
    
    Staff-level detail:
    Uses a small CNN on the mel spectrogram.
    In production, you would use a pre-trained audio model like
    AST (Audio Spectrogram Transformer) or VGGish.
    """
    def __init__(self, output_dim=256):
        super().__init__()
        self.features = nn.Sequential(
            # Input: 1 x n_mels x n_time_frames
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.projection = nn.Linear(128, output_dim)
    
    def forward(self, mel_spec):
        # mel_spec: (batch, 1, n_mels, n_time)
        features = self.features(mel_spec).squeeze(-1).squeeze(-1)
        output = self.projection(features)
        return F.normalize(output, p=2, dim=1)


audio_encoder = AudioEncoder(output_dim=256)

# Test with synthetic mel spectrogram
dummy_mel = torch.randn(4, 1, 64, 94)  # batch=4, 1 channel, 64 mels, 94 time frames
audio_emb = audio_encoder(dummy_mel)

print("=== Audio Encoder ===")
print(f"  Input:  {dummy_mel.shape} (batch, channels, mels, time_frames)")
print(f"  Output: {audio_emb.shape} (batch, 256)")
print(f"  Params: {sum(p.numel() for p in audio_encoder.parameters()):,}")
print(f"\n  In production: Use pre-trained AST or VGGish, then fine-tune.")

## 4. Fusion Strategies: Combining Modalities

Now we have three separate embeddings for each video:
- **Visual embedding** (256-dim) -- what the video *looks* like
- **Audio embedding** (256-dim) -- what the video *sounds* like
- **Text embedding** (256-dim) -- what the video's metadata *says*

How do we combine them into a single video embedding? Three approaches:

### The Restaurant Analogy

Imagine three food critics reviewing a restaurant:

1. **Early Fusion**: All three critics sit together, share notes, and write ONE review together from the start.
2. **Late Fusion**: Each critic writes their own review independently, then a manager averages the scores.
3. **Cross-Attention Fusion**: Each critic reads the others' notes and revises their own review based on what they learned.

In [ ]:
# === Fusion Strategy 1: Early Fusion ===

class EarlyFusion(nn.Module):
    """
    Concatenate raw features from all modalities, then process together.
    
    12-year-old version:
    Mix all the ingredients together at the start, then cook them.
    The model learns how to combine visual + audio + text from scratch.
    
    Pros: Can learn complex cross-modal interactions early
    Cons: Needs lots of data; computationally expensive; hard to pre-train
    """
    def __init__(self, visual_dim=256, audio_dim=256, text_dim=256, output_dim=256):
        super().__init__()
        total_dim = visual_dim + audio_dim + text_dim
        self.fuser = nn.Sequential(
            nn.Linear(total_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, output_dim)
        )
    
    def forward(self, visual_emb, audio_emb, text_emb):
        # Concatenate all modalities
        combined = torch.cat([visual_emb, audio_emb, text_emb], dim=1)  # (batch, 768)
        output = self.fuser(combined)  # (batch, 256)
        return F.normalize(output, p=2, dim=1)


# === Fusion Strategy 2: Late Fusion ===

class LateFusion(nn.Module):
    """
    Each modality is processed independently, then combined at the end.
    
    12-year-old version:
    Each chef prepares their dish separately. At the end,
    a manager tastes all three and gives a combined score.
    
    Pros: Simple; each modality can be pre-trained independently
    Cons: Misses cross-modal interactions (audio+visual combined meaning)
    """
    def __init__(self, visual_dim=256, audio_dim=256, text_dim=256, output_dim=256):
        super().__init__()
        # Learn a weight for each modality
        self.visual_weight = nn.Parameter(torch.tensor(0.5))
        self.audio_weight = nn.Parameter(torch.tensor(0.25))
        self.text_weight = nn.Parameter(torch.tensor(0.25))
        
        # Project each to same dim if needed
        self.visual_proj = nn.Linear(visual_dim, output_dim)
        self.audio_proj = nn.Linear(audio_dim, output_dim)
        self.text_proj = nn.Linear(text_dim, output_dim)
    
    def forward(self, visual_emb, audio_emb, text_emb):
        # Project each modality
        v = self.visual_proj(visual_emb)
        a = self.audio_proj(audio_emb)
        t = self.text_proj(text_emb)
        
        # Weighted sum
        weights = F.softmax(torch.stack([self.visual_weight, self.audio_weight, self.text_weight]), dim=0)
        combined = weights[0] * v + weights[1] * a + weights[2] * t
        
        return F.normalize(combined, p=2, dim=1)


# === Fusion Strategy 3: Cross-Attention Fusion ===

class CrossAttentionFusion(nn.Module):
    """
    Each modality attends to the others to enrich its representation.
    
    12-year-old version:
    Each chef looks at the other chefs' dishes and uses that
    inspiration to improve their own. Then all improved dishes
    are combined.
    
    Example: The visual embedding for a concert video attends to
    the audio embedding and realizes 'oh, there is rock music playing'
    which changes how the visual content is interpreted.
    
    Pros: Captures rich cross-modal interactions
    Cons: More parameters; slower; needs more training data
    """
    def __init__(self, embed_dim=256, n_heads=4, output_dim=256):
        super().__init__()
        # Visual attends to audio+text
        self.v_cross_attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        # Audio attends to visual+text
        self.a_cross_attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        # Text attends to visual+audio
        self.t_cross_attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        
        self.output_proj = nn.Linear(embed_dim * 3, output_dim)
    
    def forward(self, visual_emb, audio_emb, text_emb):
        # Stack modalities as sequences: each modality = 1 "token"
        # For cross-attention: query = one modality, key/value = others
        
        # Reshape to (batch, 1, dim) for attention
        v = visual_emb.unsqueeze(1)
        a = audio_emb.unsqueeze(1)
        t = text_emb.unsqueeze(1)
        
        # Visual attends to audio + text
        other_va = torch.cat([a, t], dim=1)  # (batch, 2, dim)
        v_enriched, _ = self.v_cross_attn(v, other_va, other_va)  # (batch, 1, dim)
        
        # Audio attends to visual + text
        other_at = torch.cat([v, t], dim=1)
        a_enriched, _ = self.a_cross_attn(a, other_at, other_at)
        
        # Text attends to visual + audio
        other_tv = torch.cat([v, a], dim=1)
        t_enriched, _ = self.t_cross_attn(t, other_tv, other_tv)
        
        # Concatenate enriched representations
        combined = torch.cat([
            v_enriched.squeeze(1),
            a_enriched.squeeze(1),
            t_enriched.squeeze(1)
        ], dim=1)  # (batch, dim*3)
        
        output = self.output_proj(combined)
        return F.normalize(output, p=2, dim=1)


print("=== Three Fusion Strategies ===")
for name, cls in [('Early Fusion', EarlyFusion), ('Late Fusion', LateFusion), ('Cross-Attention', CrossAttentionFusion)]:
    model = cls()
    params = sum(p.numel() for p in model.parameters())
    print(f"\n  {name}: {params:,} parameters")

In [ ]:
# === Compare Fusion Strategies ===

torch.manual_seed(42)
batch_size = 8
emb_dim = 256

# Simulate modality embeddings
visual_emb = F.normalize(torch.randn(batch_size, emb_dim), dim=1)
audio_emb = F.normalize(torch.randn(batch_size, emb_dim), dim=1)
text_meta_emb = F.normalize(torch.randn(batch_size, emb_dim), dim=1)

# Apply each fusion strategy
early_model = EarlyFusion()
late_model = LateFusion()
cross_model = CrossAttentionFusion()

early_out = early_model(visual_emb, audio_emb, text_meta_emb)
late_out = late_model(visual_emb, audio_emb, text_meta_emb)
cross_out = cross_model(visual_emb, audio_emb, text_meta_emb)

# Visualize the different outputs
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, output) in zip(axes, [
    ('Early Fusion', early_out),
    ('Late Fusion', late_out),
    ('Cross-Attention Fusion', cross_out)
]):
    # Show pairwise cosine similarities between videos in the batch
    sim = (output @ output.T).detach().numpy()
    im = ax.imshow(sim, cmap='RdYlBu_r', vmin=-1, vmax=1)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Video Index')
    ax.set_ylabel('Video Index')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Pairwise Video Similarity After Each Fusion Strategy', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Each matrix shows how similar the 8 videos are to each other.")
print("Diagonal is always 1.0 (each video is identical to itself).")
print("Different fusion strategies create different similarity structures.")

# Summary table
print("\n=== Fusion Strategy Comparison ===")
print(f"{'Strategy':<25} {'Params':>10} {'Cross-Modal?':>15} {'Best For':>25}")
print("-" * 80)
print(f"{'Early Fusion':<25} {sum(p.numel() for p in early_model.parameters()):>10,} {'Yes (implicit)':>15} {'Large datasets':>25}")
print(f"{'Late Fusion':<25} {sum(p.numel() for p in late_model.parameters()):>10,} {'No':>15} {'Simple, modular':>25}")
print(f"{'Cross-Attention':<25} {sum(p.numel() for p in cross_model.parameters()):>10,} {'Yes (explicit)':>15} {'Rich interactions':>25}")
print(f"\nRecommendation: Start with Late Fusion (simple), upgrade to Cross-Attention if needed.")

## 5. Putting It All Together: Full Multimodal Video Encoder

Now we combine everything into a single model that takes a video (frames + audio + metadata) and produces one embedding that lives in the same space as text query embeddings.

In [ ]:
# === Full Multimodal Video Encoder ===

class MultimodalVideoEncoder(nn.Module):
    """
    Complete multimodal video encoder that fuses visual, audio, and text features.
    
    12-year-old version:
    This model has three 'brains':
    - The 'eye brain' looks at video frames
    - The 'ear brain' listens to the audio
    - The 'reading brain' reads the title and description
    Then a 'combining brain' merges all three opinions into
    one final summary (embedding) of the video.
    
    This embedding lives in the SAME space as query embeddings,
    so we can compare them with dot product.
    """
    def __init__(self, vocab_size=5000, frame_dim=256, output_dim=256,
                 fusion_type='cross_attention'):
        super().__init__()
        
        # Individual modality encoders
        self.frame_encoder = SimplePatchEncoder(output_dim=output_dim)
        self.audio_encoder = AudioEncoder(output_dim=output_dim)
        self.text_encoder = MiniTransformerEncoder(vocab_size=vocab_size, embed_dim=128, output_dim=output_dim)
        
        # Temporal aggregation for frames
        self.video_encoder = VideoEncoder(frame_dim=output_dim, output_dim=output_dim)
        
        # Fusion
        if fusion_type == 'early':
            self.fusion = EarlyFusion(output_dim, output_dim, output_dim, output_dim)
        elif fusion_type == 'late':
            self.fusion = LateFusion(output_dim, output_dim, output_dim, output_dim)
        else:
            self.fusion = CrossAttentionFusion(output_dim, output_dim=output_dim)
    
    def forward(self, frames, mel_spec, title_tokens):
        """
        frames: (batch, n_frames, 3, 64, 64) - sampled video frames
        mel_spec: (batch, 1, n_mels, n_time) - audio mel spectrogram
        title_tokens: (batch, seq_len) - tokenized title/description
        """
        batch_size, n_frames = frames.shape[0], frames.shape[1]
        
        # Encode each frame individually
        frame_list = []
        for i in range(n_frames):
            frame_emb = self.frame_encoder(frames[:, i])  # (batch, 256)
            frame_list.append(frame_emb)
        frame_features = torch.stack(frame_list, dim=1)  # (batch, n_frames, 256)
        
        # Temporal aggregation
        visual_emb, _ = self.video_encoder(frame_features)  # (batch, 256)
        
        # Audio encoding
        audio_emb = self.audio_encoder(mel_spec)  # (batch, 256)
        
        # Text encoding (title/description)
        text_emb = self.text_encoder(title_tokens)  # (batch, 256)
        
        # Fusion
        fused = self.fusion(visual_emb, audio_emb, text_emb)  # (batch, 256)
        
        return fused


# Instantiate
multimodal_encoder = MultimodalVideoEncoder(vocab_size=5000, fusion_type='cross_attention')

# Test forward pass
torch.manual_seed(42)
batch = 2
test_frames = torch.randn(batch, 4, 3, 64, 64)   # 2 videos, 4 frames each
test_audio = torch.randn(batch, 1, 64, 94)         # mel spectrograms
test_titles = torch.randint(1, 5000, (batch, 10))   # tokenized titles

video_embedding = multimodal_encoder(test_frames, test_audio, test_titles)

print("=== Full Multimodal Video Encoder ===")
print(f"\n  Inputs:")
print(f"    Frames:     {test_frames.shape} (batch, n_frames, C, H, W)")
print(f"    Audio:      {test_audio.shape} (batch, 1, n_mels, n_time)")
print(f"    Title:      {test_titles.shape} (batch, seq_len)")
print(f"\n  Output:")
print(f"    Video embedding: {video_embedding.shape} (batch, 256)")
print(f"    Norm: {video_embedding.norm(dim=1).tolist()}")
print(f"\n  Total params: {sum(p.numel() for p in multimodal_encoder.parameters()):,}")
print(f"\n  This 256-dim embedding captures EVERYTHING about the video:")
print(f"  what it looks like, sounds like, and what its title says.")
print(f"  It lives in the same space as query embeddings for dot-product search.")

In [ ]:
# === Visualize the Shared Embedding Space ===

from sklearn.decomposition import PCA

# Simulate a shared embedding space with queries and videos
np.random.seed(42)

# Create clusters: each cluster = one topic
topics = ['Cats', 'Cooking', 'Sports', 'Music', 'Tech']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
n_per_topic = 10

all_embeddings = []
all_labels = []
all_types = []  # 'query' or 'video'

for i, topic in enumerate(topics):
    # Cluster center
    center = np.random.randn(256) * 0.3
    center[i*50:(i+1)*50] += 2  # Make clusters separable
    
    # Queries near center
    for _ in range(n_per_topic):
        emb = center + np.random.randn(256) * 0.3
        all_embeddings.append(emb)
        all_labels.append(i)
        all_types.append('query')
    
    # Videos near center (slightly offset)
    for _ in range(n_per_topic):
        emb = center + np.random.randn(256) * 0.3
        all_embeddings.append(emb)
        all_labels.append(i)
        all_types.append('video')

all_embeddings = np.array(all_embeddings)
all_labels = np.array(all_labels)
all_types = np.array(all_types)

# PCA to 2D for visualization
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(all_embeddings)

fig, ax = plt.subplots(figsize=(10, 8))

for i, topic in enumerate(topics):
    # Queries = circles
    query_mask = (all_labels == i) & (all_types == 'query')
    ax.scatter(embeddings_2d[query_mask, 0], embeddings_2d[query_mask, 1],
              c=colors[i], marker='o', s=80, alpha=0.7, edgecolors='black', linewidth=0.5,
              label=f'{topic} (queries)')
    
    # Videos = triangles
    video_mask = (all_labels == i) & (all_types == 'video')
    ax.scatter(embeddings_2d[video_mask, 0], embeddings_2d[video_mask, 1],
              c=colors[i], marker='^', s=100, alpha=0.7, edgecolors='black', linewidth=0.5,
              label=f'{topic} (videos)')

ax.set_xlabel('PCA Component 1', fontsize=12)
ax.set_ylabel('PCA Component 2', fontsize=12)
ax.set_title('Shared Embedding Space: Queries (circles) and Videos (triangles)\n'
             'Same-topic queries and videos cluster together!', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("This is the goal of contrastive learning:")
print("- Queries about cats cluster near cat videos")
print("- Queries about cooking cluster near cooking videos")
print("- At search time, we find the nearest video neighbors of the query embedding")

## Key Takeaways

1. **Text embeddings evolved** from Bag of Words (sparse, no semantics) to BERT (dense, context-aware). For video search, we use Transformer-based encoders.

2. **Video features** are extracted by sampling frames (8 is enough), encoding each with ViT, and aggregating with temporal attention. Frame-level is much cheaper than video-level processing.

3. **Audio features** via mel spectrograms turn audio into a 2D image that can be processed by standard CNNs. This captures music type, speech, and sound effects.

4. **Three fusion strategies**: Early fusion (mix raw features), Late fusion (combine final embeddings), Cross-attention (modalities interact). Start simple (late), upgrade as needed (cross-attention).

5. **Shared embedding space** is the key idea: both queries and videos are mapped to the same vector space, so search becomes a nearest-neighbor lookup.

6. **Modality importance varies by query**: "acoustic guitar" needs audio features; "sunset beach" needs visual; "Taylor Swift" needs text metadata. Cross-attention lets the model learn when to weight each modality.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)